|<h2>Course:</h2>|<h1><a href="https://derivingsystems.com/course.html" target="_blank">Build your own vLLM: inference engines from the memory system up</a></h1>|
|-|:-:|
|<h2>Part 8:</h2>|<h1>The Capstone<h1>|
|<h2>Section:</h2>|<h1>Kernel counters<h1>|
|<h2>Lecture:</h2>|<h1><b>CodeChallenge: read the counters<b></h1>|

<br>

<h5><b>Course repo:</b> <a href="https://github.com/Venugopalan2610/vllm-from-scratch" target="_blank">github.com/Venugopalan2610/vllm-from-scratch</a></h5>
<h5><b>The derivations:</b> <a href="https://derivingsystems.com" target="_blank">derivingsystems.com</a></h5>
<i>The notebooks build the intuition. The ladder in app/ makes you build the thing.</i>

In [ ]:
# find the repo root. The directory you start from does not matter.
import sys, time, shutil
from pathlib import Path
ROOT = next(folder for folder in [Path.cwd(), *Path.cwd().parents]
            if (folder/'cudalib').is_dir())
sys.path.insert(0, str(ROOT))
import torch
import cudalib
WORK = ROOT / '.cudacache' / 'practicum_j'
WORK.mkdir(parents=True, exist_ok=True)

# Read the counters

A clock tells you that a kernel is slow. The hardware counters tell you
**why**: the bytes that the memory delivered, the part of each fetched sector
that the kernel used, the bank conflicts in [shared memory](../../GLOSSARY.md#shared-memory), and the warps that
were resident.

This notebook compiles three small kernels that you already met in the
incidents of Parts 3 and 8:

- **A**: one thread for each row of 128 bf16 values (the uncoalesced read of
  stage 08);
- **B**: 16 threads for each row, 16 bytes each (the coalesced read of stage
  08b);
- **C**: a tile of 32 x 128 floats in shared memory, where the lanes of a warp
  read one column, and the same with rows of 129 floats.

First you measure them with a clock, which always works. Then you read their
counters with **Nsight Compute** (`ncu`), which needs a permission that many
Linux machines do not give a normal user. The notebook tells you if yours does
not, and how to change it. The checks at the end always check the clock, and
they check the counters when they can.

In [ ]:
KERNELS = r"""
#include <ATen/cuda/CUDAContext.h>
#include <torch/extension.h>
#include <cuda_bf16.h>

// A: one thread for each row of 128 bf16 values. Each thread walks its row.
__global__ void rows_by_thread(const __nv_bfloat16* keys, float* out, long rows) {
  long row = blockIdx.x * (long)blockDim.x + threadIdx.x;
  if (row >= rows) return;
  float sum = 0.f;
  for (int d = 0; d < 128; ++d) sum += __bfloat162float(keys[row * 128 + d]);
  out[row] = sum;
}

// B: 16 threads for each row, 16 bytes each.
__global__ void rows_by_16_threads(const __nv_bfloat16* keys, float* out, long rows) {
  long lane = threadIdx.x % 16;
  for (long row = (blockIdx.x * (long)blockDim.x + threadIdx.x) / 16; row < rows;
       row += (long)gridDim.x * blockDim.x / 16) {
    uint4 chunk = reinterpret_cast<const uint4*>(keys + row * 128)[lane];
    const __nv_bfloat16* v = reinterpret_cast<const __nv_bfloat16*>(&chunk);
    float sum = 0.f;
    for (int i = 0; i < 8; ++i) sum += __bfloat162float(v[i]);
    for (int offset = 8; offset > 0; offset /= 2) sum += __shfl_down_sync(0xffffffff, sum, offset, 16);
    if (lane == 0) out[row] = sum;
  }
}

// C: a tile of 32 rows in shared memory, and each lane of a warp reads its own row, one column.
template <int WIDTH>
__device__ void tile_columns(float* out, int repeats) {
  __shared__ float tile[32][WIDTH];
  int lane = threadIdx.x;
  for (int c = 0; c < 128; ++c) tile[lane][c] = lane * 0.5f + c;
  __syncwarp();
  float sum = 0.f;
  for (int r = 0; r < repeats; ++r)
    for (int c = 0; c < 128; ++c) sum += tile[lane][(c + r) & 127];
  out[blockIdx.x * 32 + lane] = sum;
}
__global__ void tile_rows_128(float* out, int repeats) { tile_columns<128>(out, repeats); }
__global__ void tile_rows_129(float* out, int repeats) { tile_columns<129>(out, repeats); }

void run_a(torch::Tensor keys, torch::Tensor out) {
  long rows = keys.size(0);
  rows_by_thread<<<(rows + 255) / 256, 256, 0, at::cuda::getCurrentCUDAStream()>>>(
      (const __nv_bfloat16*)keys.data_ptr(), out.data_ptr<float>(), rows);
}
void run_b(torch::Tensor keys, torch::Tensor out) {
  rows_by_16_threads<<<1 << 16, 256, 0, at::cuda::getCurrentCUDAStream()>>>(
      (const __nv_bfloat16*)keys.data_ptr(), out.data_ptr<float>(), keys.size(0));
}
void run_c(torch::Tensor out, int64_t width, int64_t repeats) {
  int blocks = out.numel() / 32;
  auto stream = at::cuda::getCurrentCUDAStream();
  if (width == 128) tile_rows_128<<<blocks, 32, 0, stream>>>(out.data_ptr<float>(), repeats);
  else tile_rows_129<<<blocks, 32, 0, stream>>>(out.data_ptr<float>(), repeats);
}
PYBIND11_MODULE(TORCH_EXTENSION_NAME, m) {
  m.def("run_a", &run_a); m.def("run_b", &run_b); m.def("run_c", &run_c);
}
"""
kernels = cudalib.build_source('practicum_j', KERNELS)
keys = torch.randn(1 << 20, 128, device='cuda', dtype=torch.bfloat16)     # 268 MB, far more than L2
out = torch.empty(1 << 20, device='cuda')
tile_out = torch.empty(4096 * 32, device='cuda')
PEAK = cudalib.peak_bandwidth()
print(f'the kernels are built. The streaming copy of this card: {PEAK:.0f} GB/s')

# Exercise 1: the clock

Measure A and B in GB/s, and the two versions of C in ms. Before you run it,
predict the ratio B / A and the ratio of C with rows of 128 to C with rows of
129.

In [ ]:
def gbs(fn, num_bytes):
    return num_bytes / (cudalib.bench_ms(fn) / 1000) / 1e9

a = gbs(lambda: kernels.run_a(keys, out), keys.numel() * 2)
b = gbs(lambda: kernels.run_b(keys, out), keys.numel() * 2)
c128 = cudalib.bench_ms(lambda: kernels.run_c(tile_out, 128, 64))
c129 = cudalib.bench_ms(lambda: kernels.run_c(tile_out, 129, 64))
print(f'A {a:.0f} GB/s, B {b:.0f} GB/s ({b / a:.2f}x);  C with rows of 128: {c128:.2f} ms, of 129: {c129:.2f} ms ({c128 / c129:.1f}x)')

# Exercise 2: can this machine read the counters?

Run this cell as it is. It writes a small script that runs the four kernels,
and runs Nsight Compute on it.

In [ ]:
METRICS = {
    'dram__bytes.sum.per_second': 'bytes out of DRAM each second',
    'smsp__average_data_bytes_per_sector_mem_global_op_ld.pct': 'of each 32-byte sector fetched, the % used',
    'l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum': 'shared-memory bank conflicts on loads',
    'smsp__inst_executed_op_shared_ld.sum': 'shared-memory load instructions',
    'sm__warps_active.avg.pct_of_peak_sustained_active': 'achieved occupancy',
}
DRIVER = WORK / 'driver.py'
DRIVER.write_text(f"""
import sys, torch
sys.path.insert(0, {str(ROOT)!r})
import cudalib
KERNELS = {KERNELS!r}
kernels = cudalib.build_source('practicum_j', KERNELS)
keys = torch.randn(1 << 20, 128, device='cuda', dtype=torch.bfloat16)
out = torch.empty(1 << 20, device='cuda')
tile_out = torch.empty(4096 * 32, device='cuda')
kernels.run_a(keys, out); kernels.run_b(keys, out)
kernels.run_c(tile_out, 128, 64); kernels.run_c(tile_out, 129, 64)
torch.cuda.synchronize()
""")

def counters(kernel_regex):
    """-> {metric: float} for the first kernel that matches, or None."""
    values, output = cudalib.run_ncu(DRIVER, list(METRICS), kernel=f'regex:{kernel_regex}')
    if not values:
        return None, output
    return {name: float(str(value).replace(',', '')) for name, (value, _unit) in values.items()}, output

first, output = counters('rows_by_thread')
HAVE_COUNTERS = first is not None
if not HAVE_COUNTERS:
    print('Nsight Compute cannot read the counters here.')
    if cudalib.probe.NO_COUNTER_PERMISSION in output:
        print("The driver lets only an administrator read them. To change it:\n"
              "    echo 'options nvidia NVreg_RestrictProfilingToAdminUsers=0' | sudo tee /etc/modprobe.d/nvidia-profiling.conf\n"
              "    sudo update-initramfs -u     # then reboot\n"
              "Until then, the checks of this notebook use the clock only.")
    else:
        print(output[-1500:])
else:
    print('Nsight Compute can read the counters.')

# Exercise 3: the counters, and what they mean

Write three functions that turn the raw counters into the three numbers that
explain Exercise 1. Before you run it, predict each number for A, B and the two
versions of C.

In [ ]:
if HAVE_COUNTERS:
    results = {name: counters(regex)[0] for name, regex in
               [('A', 'rows_by_thread'), ('B', 'rows_by_16_threads'),
                ('C128', 'tile_rows_128'), ('C129', 'tile_rows_129')]}
    for name, values in results.items():
        print(name, {metric.split('__')[1][:40]: round(value, 2) for metric, value in values.items()})

def sector_use(values):
    return values['smsp__average_data_bytes_per_sector_mem_global_op_ld.pct']

def conflicts_per_load(values):
    loads = values['smsp__inst_executed_op_shared_ld.sum']
    return values['l1tex__data_bank_conflicts_pipe_lsu_mem_shared_op_ld.sum'] / max(loads, 1)

def dram_fraction(values):
    return values['dram__bytes.sum.per_second'] / (PEAK * 1e9)

### The checks

The clock checks always run. The counter checks run when Nsight Compute can
read the counters.

In [ ]:
### THE CHECKS. Do not edit this cell.

assert all(value > 0 for value in (a, b, c128, c129)), 'run Exercise 1 first'
assert b > 1.25 * a, f'B must stream faster than A: {b:.0f} against {a:.0f} GB/s'
assert c128 > 4 * c129, f'rows of 128 floats must be much slower: only {c128 / c129:.1f}x'
if HAVE_COUNTERS:
    assert sector_use(results['A']) < 40, 'A uses a small part of each sector'
    assert sector_use(results['B']) > 90, 'B uses almost all of each sector'
    assert conflicts_per_load(results['C128']) > 20, 'rows of 128: about 31 conflicts per load'
    assert conflicts_per_load(results['C129']) < 1, 'rows of 129: no conflict'
    assert dram_fraction(results['B']) > dram_fraction(results['A'])
    print('All the checks pass, with the counters and with the clock.')
else:
    print('The checks with the clock pass. The counter checks did not run: '
          'this machine does not let this user read the counters.')

# Exercise 4: your kernels (after stage 08b)

`./vc ncu 8 64` and `./vc ncu 8b 64` print the same counters for your paged
attention kernels. Answer:

1. What part of each sector does your stage 08 kernel use, and your stage 08b
   kernel? Does the ratio match the gain that stage 08b measured?
2. Is your stage 08b kernel near the bandwidth of the card? If not, which
   counter tells you why?

### Before you open the solution

1. A reads 2 bytes of each 32-byte sector, 6.25%. Why is A not 16 times slower
   than B?
2. Kernel C with rows of 128 floats has about 31 conflicts for each load. Why
   31, and not 32?
3. Why does the notebook check the clock even when the counters work?

### What I measured

On an RTX 4080 Laptop GPU with the clock: A about 210 GB/s, B about 320 GB/s
(1.5x); C with rows of 128 about 8 ms, with rows of 129 about 0.5 ms (15x). This
machine does not let a normal user read the counters, so I could not run the
counter cells here. With the counters, expect a sector use near 6% for A (the L1
cache serves the rest of each sector to the next loads of the same thread, which
is why A is not 16x slower), near 100% for B, about 31 conflicts for each load
for C with rows of 128 (32 accesses to one bank: one of them is not a conflict),
and 0 with rows of 129.